In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.applications import VGG16, ResNet50V2
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("Drive mounted!")

Mounted at /content/drive
Drive mounted!


In [ ]:
import shutil
import os

source = '/content/drive/MyDrive/emotion_dataset/Dataset'
destination = '/content/dataset'

if os.path.exists(destination):
    print("Dataset already copied!")
else:
    shutil.copytree(source, destination)
    print(" Dataset copied to local storage!")

train_count = sum([len(os.listdir(f'/content/dataset/train/{e}')) for e in os.listdir('/content/dataset/train')])
test_count = sum([len(os.listdir(f'/content/dataset/test/{e}')) for e in os.listdir('/content/dataset/test')])

print(f"\n Local dataset ready:")
print(f"   Train: {train_count} images")
print(f"   Test: {test_count} images")

 Dataset copied to local storage!

 Local dataset ready:
   Train: 28304 images
   Test: 7067 images


In [ ]:
import os

train_path = '/content/dataset/train'
test_path = '/content/dataset/test'

if os.path.exists(train_path):
    classes = sorted(os.listdir(train_path))
    print(f"Classes: {classes}")

    total_train = 0
    for cls in classes:
      cls_path = os.path.join(train_path, cls)
      if os.path.isdir(cls_path):
        num_files = len(os.listdir(cls_path))
        total_train += num_files
        print(f"  {cls}: {num_files} images")

    print(f"\nTotal training images: {total_train}")
else:
  print('Train path does not exist')

if os.path.exists(test_path):
  total_test = 0
  for cls in sorted(os.listdir(test_path)):
    cls_path = os.path.join(test_path, cls)
    if os.path.isdir(cls_path):
      num_files = len(os.listdir(cls_path))
      total_test += num_files
      print(f"  {cls}: {num_files} images")
else:
  print('Test path does not exist')

print(f"\nTotal testing images: {total_test}")

Classes: ['angry', 'fear', 'happy', 'neutral', 'sad', 'surprise']
  angry: 3995 images
  fear: 4097 images
  happy: 7245 images
  neutral: 4966 images
  sad: 4830 images
  surprise: 3171 images

Total training images: 28304
  angry: 958 images
  fear: 1024 images
  happy: 1774 images
  neutral: 1233 images
  sad: 1247 images
  surprise: 831 images

Total testing images: 7067


In [ ]:
IMG_SIZE = 48
BATCH_SIZE = 32
EPOCHS = 40
NUM_CLASSES = 6
EMOTIONS = ['angry', 'fear', 'happy', 'neutral', 'sad', 'surprise']

In [ ]:

train_datagen = ImageDataGenerator(
    rescale = 1./255,
    rotation_range = 25,
    width_shift_range = 0.2,
    height_shift_range = 0.2,
    zoom_range = 0.2,
    horizontal_flip = True,
    shear_range = 0.15,
    brightness_range = [0.8, 1.2],
    fill_mode = 'nearest'
)

test_datagen = ImageDataGenerator(
    rescale = 1./255
)

train_generator = train_datagen.flow_from_directory(
    '/content/dataset/train',
    target_size = (IMG_SIZE, IMG_SIZE),
    color_mode = 'grayscale',
    batch_size = BATCH_SIZE,
    class_mode = 'categorical',
    shuffle = True
)

test_generator = test_datagen.flow_from_directory(
   '/content/dataset/test',
    target_size = (IMG_SIZE, IMG_SIZE),
    color_mode = 'grayscale',
    batch_size = BATCH_SIZE,
    class_mode = 'categorical',
    shuffle = False
)
print(f"\n Training batches: {len(train_generator)} ")
print(f"\n Testing batches: {len(test_generator)} ")
print(f" Images per batch: {BATCH_SIZE}")
print(f"\n class mapping: {train_generator.class_indices}")


Found 28304 images belonging to 6 classes.
Found 7067 images belonging to 6 classes.

 Training batches: 885 

 Testing batches: 221 
 Images per batch: 32

 class mapping: {'angry': 0, 'fear': 1, 'happy': 2, 'neutral': 3, 'sad': 4, 'surprise': 5}


In [ ]:
#Model 1
def build_cnn():
  model = models.Sequential([
      #Block 1
      layers.Conv2D(64, (3,3), activation = 'relu', input_shape = (48,48,1), padding = 'same'),
      layers.BatchNormalization(),
      layers.Conv2D(64, (3,3), activation = 'relu', padding = 'same'),
      layers.BatchNormalization(),
      layers.MaxPooling2D((2,2)),
      layers.Dropout(0.25),

      #Block 2
      layers.Conv2D(128, (3,3), activation = 'relu', padding = 'same'),
      layers.BatchNormalization(),
      layers.Conv2D(128, (3,3), activation = 'relu', padding = 'same'),
      layers.BatchNormalization(),
      layers.MaxPooling2D((2,2)),
      layers.Dropout(0.25),

      #Block 3
      layers.Conv2D(256, (3,3), activation = 'relu', padding = 'same'),
      layers.BatchNormalization(),
      layers.Conv2D(256, (3,3), activation = 'relu', padding = 'same'),
      layers.BatchNormalization(),
      layers.MaxPooling2D((2,2)),
      layers.Dropout(0.3),

      #Block 4
      layers.Conv2D(512, (3,3), activation = 'relu', padding = 'same'),
      layers.BatchNormalization(),
      layers.MaxPooling2D((2,2)),
      layers.Dropout(0.3),

      #Dense Layers
      layers.Flatten(),
      layers.Dense(512, activation = 'relu'),
      layers.BatchNormalization(),
      layers.Dropout(0.5),
      layers.Dense(256, activation = 'relu'),
      layers.Dropout(0.5),
      layers.Dense(NUM_CLASSES, activation = 'softmax')
  ])

  return model

In [ ]:
def build_vgg16():
  inputs = layers.Input(shape = (IMG_SIZE, IMG_SIZE, 1))
  x = layers.Conv2D(3, (1,1), padding = 'same')(inputs)

  base_model = VGG16(weights = 'imagenet', include_top = False, input_shape = (48, 48, 3))

  for layer in base_model.layers:
    layer.trainable = False

  x = base_model(x)
  x = layers.GlobalAveragePooling2D()(x)
  x = layers.Dense(512, activation = 'relu')(x)
  x = layers.BatchNormalization()(x)
  x = layers.Dropout(0.5)(x)
  x = layers.Dense(256, activation = 'relu')(x)
  x = layers.Dropout(0.4)(x)
  outputs = layers.Dense(NUM_CLASSES, activation = 'softmax')(x)

  return models.Model(inputs = inputs, outputs = outputs)

In [ ]:
from IPython.core import history

def build_resnet50():
  inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1))
  x = layers.Conv2D(3, (1,1), padding = 'same')(inputs)

  base_model = ResNet50V2(weights = 'imagenet', include_top = False, input_shape = (48, 48, 3))

  for layer in base_model.layers[:-20]:
    layer.trainable = False

  x = base_model(x)
  x = layers.GlobalAveragePooling2D()(x)
  x = layers.Dense(512, activation = 'relu')(x)
  x = layers.BatchNormalization()(x)
  x = layers.Dropout(0.5)(x)
  x = layers.Dense(256, activation = 'relu')(x)
  x = layers.Dropout(0.4)(x)
  outputs = layers.Dense(NUM_CLASSES, activation = 'softmax')(x)

  return models.Model(inputs = inputs, outputs = outputs)


In [ ]:
from IPython.testing import test
def train_model(model, model_name):
  print(f"Training: {model_name}")
  model.compile(
      optimizer = Adam(learning_rate = 0.0001),
      loss = 'categorical_crossentropy',
      metrics = ['accuracy']
  )

  os.makedirs("saved_models", exist_ok = True)

  callbacks = [
      ModelCheckpoint(
          f'saved_models/{model_name}.h5',
          monitor = 'val_accuracy',
          save_best_only = True,
          mode = 'max',
          verbose = 1
      ),

      EarlyStopping(
          monitor = 'val_loss',
          patience = 10,
          restore_best_weights = True,
          verbose = 1
      ),

      ReduceLROnPlateau(
          monitor = 'val_loss',
          factor = 0.5,
          patience = 5,
          min_lr = 1e-7,
          verbose = 1  )
    ]

  history = model.fit(
      train_generator,
      validation_data = test_generator,
      epochs = EPOCHS,
      callbacks = callbacks,
      verbose = 1
  )

  test_loss, test_acc = model.evaluate(test_generator, verbose = 0)

  print(f"{model_name}, Training Complete")
  print(f"Test Loss: {test_loss}")
  print(f"Test Accuracy: {test_acc}")

  return history, test_acc

In [ ]:
results = {}
histories = {}
print("\n Model 1/3: CNN")
model1 = build_cnn()
history1 , acc1 = train_model(model1, 'cnn')
results['cnn'] = acc1
histories['cnn'] = history1




 Model 1/3: CNN
Training: cnn


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.1754 - loss: 2.9517
Epoch 1: val_accuracy improved from -inf to 0.26107, saving model to saved_models/cnn.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 59s 51ms/step - accuracy: 0.1755 - loss: 2.9510 - val_accuracy: 0.2611 - val_loss: 1.7833 - learning_rate: 1.0000e-04
Epoch 2/40
885/885 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.2045 - loss: 2.2471
Epoch 2: val_accuracy improved from 0.26107 to 0.26404, saving model to saved_models/cnn.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 35s 39ms/step - accuracy: 0.2045 - loss: 2.2470 - val_accuracy: 0.2640 - val_loss: 1.7819 - learning_rate: 1.0000e-04
Epoch 3/40
885/885 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.2060 - loss: 2.0669
Epoch 3: val_accuracy improved from 0.26404 to 0.26956, saving model to saved_models/cnn.h5


...

885/885 ━━━━━━━━━━━━━━━━━━━━ 34s 39ms/step - accuracy: 0.5913 - loss: 1.0683 - val_accuracy: 0.6363 - val_loss: 0.9490 - learning_rate: 1.0000e-04
Epoch 33/40
885/885 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.5972 - loss: 1.0593
Epoch 33: val_accuracy improved from 0.63634 to 0.63648, saving model to saved_models/cnn.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 35s 40ms/step - accuracy: 0.5972 - loss: 1.0593 - val_accuracy: 0.6365 - val_loss: 0.9459 - learning_rate: 1.0000e-04
Epoch 34/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.6011 - loss: 1.0445
Epoch 34: val_accuracy did not improve from 0.63648
885/885 ━━━━━━━━━━━━━━━━━━━━ 34s 39ms/step - accuracy: 0.6011 - loss: 1.0445 - val_accuracy: 0.6290 - val_loss: 0.9561 - learning_rate: 1.0000e-04
Epoch 35/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.6024 - loss: 1.0401
Epoch 35: val_accuracy improved from 0.63648 to 0.64143, saving model to saved_models/cnn.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 35s 40ms/step - accuracy: 0.6024 - loss: 1.0401 - val_accuracy: 0.6414 - val_loss: 0.9275 - learning_rate: 1.0000e-04
Epoch 36/40
885/885 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.6053 - loss: 1.0383
Epoch 36: val_accuracy improved from 0.64143 to 0.64242, saving model to saved_models/cnn.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 34s 39ms/step - accuracy: 0.6053 - loss: 1.0383 - val_accuracy: 0.6424 - val_loss: 0.9320 - learning_rate: 1.0000e-04
Epoch 37/40
885/885 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.5985 - loss: 1.0367
Epoch 37: val_accuracy did not improve from 0.64242
885/885 ━━━━━━━━━━━━━━━━━━━━ 35s 39ms/step - accuracy: 0.5985 - loss: 1.0367 - val_accuracy: 0.6259 - val_loss: 0.9722 - learning_rate: 1.0000e-04
Epoch 38/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.6200 - loss: 1.0041
Epoch 38: val_accuracy did not improve from 0.64242
885/885 ━━━━━━━━━━━━━━━━━━━━ 34s 39ms/step - accuracy: 0.6199 - loss: 1.0041 - val_accuracy: 0.6320 - val_loss: 0.9524 - learning_rate: 1.0000e-04
Epoch 39/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.6053 - loss: 1.0293
Epoch 39: val_accuracy improved from 0.64242 to 0.64313, saving model to saved_models/cnn.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 37s 42ms/step - accuracy: 0.6053 - loss: 1.0293 - val_accuracy: 0.6431 - val_loss: 0.9329 - learning_rate: 1.0000e-04
Epoch 40/40
885/885 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.6120 - loss: 1.0179
Epoch 40: val_accuracy did not improve from 0.64313

Epoch 40: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
885/885 ━━━━━━━━━━━━━━━━━━━━ 34s 39ms/step - accuracy: 0.6120 - loss: 1.0179 - val_accuracy: 0.6256 - val_loss: 0.9814 - learning_rate: 1.0000e-04
Restoring model weights from the end of the best epoch: 35.
cnn, Training Complete
Test Loss: 0.9275342226028442
Test Accuracy: 0.641431987285614


In [ ]:
print("\n Model 2/3: VGG16")
model2 = build_vgg16()
history2 , acc2 = train_model(model2, 'vgg16')
results['vgg16'] = acc2
histories['vgg16'] = history2


 Model 2/3: VGG16
Training: vgg16
Epoch 1/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.2016 - loss: 2.2407
Epoch 1: val_accuracy improved from -inf to 0.30324, saving model to saved_models/vgg16.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 46s 46ms/step - accuracy: 0.2016 - loss: 2.2404 - val_accuracy: 0.3032 - val_loss: 1.7102 - learning_rate: 1.0000e-04
Epoch 2/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.2282 - loss: 1.9424
Epoch 2: val_accuracy improved from 0.30324 to 0.30593, saving model to saved_models/vgg16.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 36s 41ms/step - accuracy: 0.2282 - loss: 1.9423 - val_accuracy: 0.3059 - val_loss: 1.6852 - learning_rate: 1.0000e-04
Epoch 3/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.2427 - loss: 1.8430
Epoch 3: val_accuracy improved from 0.30593 to 0.31116, saving model to saved_models/vgg16.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 37s 42ms/step - accuracy: 0.2427 - loss: 1.8429 - val_accuracy: 0.3112 - val_loss: 1.6739 - learning_rate: 1.0000e-04
Epoch 4/40
885/885 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.2468 - loss: 1.7874
Epoch 4: val_accuracy improved from 0.31116 to 0.31385, saving model to saved_models/vgg16.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 36s 41ms/step - accuracy: 0.2468 - loss: 1.7874 - val_accuracy: 0.3139 - val_loss: 1.6733 - learning_rate: 1.0000e-04
Epoch 5/40
885/885 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.2646 - loss: 1.7459
Epoch 5: val_accuracy improved from 0.31385 to 0.31569, saving model to saved_models/vgg16.h5


...

885/885 ━━━━━━━━━━━━━━━━━━━━ 36s 40ms/step - accuracy: 0.3257 - loss: 1.6342 - val_accuracy: 0.3605 - val_loss: 1.5681 - learning_rate: 1.0000e-04
Epoch 27/40
885/885 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.3235 - loss: 1.6377
Epoch 27: val_accuracy did not improve from 0.36055
885/885 ━━━━━━━━━━━━━━━━━━━━ 36s 40ms/step - accuracy: 0.3235 - loss: 1.6377 - val_accuracy: 0.3593 - val_loss: 1.5672 - learning_rate: 1.0000e-04
Epoch 28/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.3257 - loss: 1.6350
Epoch 28: val_accuracy improved from 0.36055 to 0.36366, saving model to saved_models/vgg16.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 36s 41ms/step - accuracy: 0.3257 - loss: 1.6350 - val_accuracy: 0.3637 - val_loss: 1.5612 - learning_rate: 1.0000e-04
Epoch 29/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.3349 - loss: 1.6330
Epoch 29: val_accuracy did not improve from 0.36366
885/885 ━━━━━━━━━━━━━━━━━━━━ 36s 41ms/step - accuracy: 0.3349 - loss: 1.6330 - val_accuracy: 0.3603 - val_loss: 1.5610 - learning_rate: 1.0000e-04
Epoch 30/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.3331 - loss: 1.6219
Epoch 30: val_accuracy did not improve from 0.36366
885/885 ━━━━━━━━━━━━━━━━━━━━ 35s 40ms/step - accuracy: 0.3330 - loss: 1.6219 - val_accuracy: 0.3637 - val_loss: 1.5603 - learning_rate: 1.0000e-04
Epoch 31/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.3306 - loss: 1.6262
Epoch 31: val_accuracy improved from 0.36366 to 0.36508, saving model to saved_models/vgg16.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 36s 41ms/step - accuracy: 0.3306 - loss: 1.6262 - val_accuracy: 0.3651 - val_loss: 1.5573 - learning_rate: 1.0000e-04
Epoch 32/40
885/885 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.3381 - loss: 1.6244
Epoch 32: val_accuracy improved from 0.36508 to 0.37173, saving model to saved_models/vgg16.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 36s 40ms/step - accuracy: 0.3381 - loss: 1.6244 - val_accuracy: 0.3717 - val_loss: 1.5515 - learning_rate: 1.0000e-04
Epoch 33/40
885/885 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.3290 - loss: 1.6263
Epoch 33: val_accuracy did not improve from 0.37173
885/885 ━━━━━━━━━━━━━━━━━━━━ 36s 41ms/step - accuracy: 0.3290 - loss: 1.6263 - val_accuracy: 0.3671 - val_loss: 1.5567 - learning_rate: 1.0000e-04
Epoch 34/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.3343 - loss: 1.6191
Epoch 34: val_accuracy did not improve from 0.37173
885/885 ━━━━━━━━━━━━━━━━━━━━ 35s 40ms/step - accuracy: 0.3343 - loss: 1.6191 - val_accuracy: 0.3680 - val_loss: 1.5503 - learning_rate: 1.0000e-04
Epoch 35/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.3339 - loss: 1.6193
Epoch 35: val_accuracy did not improve from 0.37173
885/885 ━━━━━━━━━━━━━━━━━━━━ 36s 41ms/step - accuracy: 0.3339 - loss: 1.6193 - val_accuracy: 0.3696 - val_loss: 1.5492 - learning_rate: 1.0

885/885 ━━━━━━━━━━━━━━━━━━━━ 36s 40ms/step - accuracy: 0.3393 - loss: 1.6152 - val_accuracy: 0.3733 - val_loss: 1.5485 - learning_rate: 1.0000e-04
Epoch 37/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.3378 - loss: 1.6195
Epoch 37: val_accuracy did not improve from 0.37328
885/885 ━━━━━━━━━━━━━━━━━━━━ 36s 40ms/step - accuracy: 0.3378 - loss: 1.6195 - val_accuracy: 0.3719 - val_loss: 1.5439 - learning_rate: 1.0000e-04
Epoch 38/40
885/885 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.3378 - loss: 1.6167
Epoch 38: val_accuracy did not improve from 0.37328
885/885 ━━━━━━━━━━━━━━━━━━━━ 35s 40ms/step - accuracy: 0.3378 - loss: 1.6168 - val_accuracy: 0.3717 - val_loss: 1.5437 - learning_rate: 1.0000e-04
Epoch 39/40
885/885 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.3413 - loss: 1.6139
Epoch 39: val_accuracy did not improve from 0.37328
885/885 ━━━━━━━━━━━━━━━━━━━━ 36s 40ms/step - accuracy: 0.3413 - loss: 1.6139 - val_accuracy: 0.3730 - val_loss: 1.5371 - learning_rate: 1.0

In [ ]:
print("\n Model 3/3: ResNet50")
model3 = build_resnet50()
history3 , acc3 = train_model(model3, 'resnet50')
results['resnet50'] = acc3
histories['resnet50'] = history3

print("All models trained!")
print("\n Result Summary:\n")
for name, acc in results.items():
  print(f"{name}: {acc}")

best_model = max(results.items(), key = lambda x: x[1])
print(f"\nBest Model: {best_model[0]} ({best_model[1]*100:.2f}%)")


 Model 3/3: ResNet50
94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training: resnet50
Epoch 1/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.1926 - loss: 2.5724
Epoch 1: val_accuracy improved from -inf to 0.28598, saving model to saved_models/resnet50.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 75s 62ms/step - accuracy: 0.1927 - loss: 2.5720 - val_accuracy: 0.2860 - val_loss: 1.8402 - learning_rate: 1.0000e-04
Epoch 2/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.2207 - loss: 2.0790
Epoch 2: val_accuracy improved from 0.28598 to 0.30055, saving model to saved_models/resnet50.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 37s 42ms/step - accuracy: 0.2207 - loss: 2.0789 - val_accuracy: 0.3006 - val_loss: 1.6887 - learning_rate: 1.0000e-04
Epoch 3/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.2452 - loss: 1.9119
Epoch 3: val_accuracy improved from 0.30055 to 0.32942, saving model to saved_models/resnet50.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 39s 44ms/step - accuracy: 0.2453 - loss: 1.9118 - val_accuracy: 0.3294 - val_loss: 1.6373 - learning_rate: 1.0000e-04
Epoch 4/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.2625 - loss: 1.8223
Epoch 4: val_accuracy improved from 0.32942 to 0.32998, saving model to saved_models/resnet50.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 36s 41ms/step - accuracy: 0.2625 - loss: 1.8222 - val_accuracy: 0.3300 - val_loss: 1.6286 - learning_rate: 1.0000e-04
Epoch 5/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.2803 - loss: 1.7625
Epoch 5: val_accuracy improved from 0.32998 to 0.34470, saving model to saved_models/resnet50.h5


...

885/885 ━━━━━━━━━━━━━━━━━━━━ 39s 44ms/step - accuracy: 0.4177 - loss: 1.4773 - val_accuracy: 0.4527 - val_loss: 1.3890 - learning_rate: 1.0000e-04
Epoch 33/40
885/885 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.4211 - loss: 1.4711
Epoch 33: val_accuracy improved from 0.45267 to 0.46045, saving model to saved_models/resnet50.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 37s 42ms/step - accuracy: 0.4211 - loss: 1.4711 - val_accuracy: 0.4604 - val_loss: 1.3807 - learning_rate: 1.0000e-04
Epoch 34/40
885/885 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.4167 - loss: 1.4669
Epoch 34: val_accuracy improved from 0.46045 to 0.46059, saving model to saved_models/resnet50.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 39s 44ms/step - accuracy: 0.4167 - loss: 1.4669 - val_accuracy: 0.4606 - val_loss: 1.3768 - learning_rate: 1.0000e-04
Epoch 35/40
885/885 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.4251 - loss: 1.4656
Epoch 35: val_accuracy did not improve from 0.46059
885/885 ━━━━━━━━━━━━━━━━━━━━ 35s 40ms/step - accuracy: 0.4251 - loss: 1.4656 - val_accuracy: 0.4505 - val_loss: 1.3844 - learning_rate: 1.0000e-04
Epoch 36/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.4299 - loss: 1.4543
Epoch 36: val_accuracy improved from 0.46059 to 0.46158, saving model to saved_models/resnet50.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 40s 45ms/step - accuracy: 0.4299 - loss: 1.4543 - val_accuracy: 0.4616 - val_loss: 1.3781 - learning_rate: 1.0000e-04
Epoch 37/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.4331 - loss: 1.4530
Epoch 37: val_accuracy improved from 0.46158 to 0.46413, saving model to saved_models/resnet50.h5


885/885 ━━━━━━━━━━━━━━━━━━━━ 37s 42ms/step - accuracy: 0.4331 - loss: 1.4530 - val_accuracy: 0.4641 - val_loss: 1.3689 - learning_rate: 1.0000e-04
Epoch 38/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.4299 - loss: 1.4501
Epoch 38: val_accuracy did not improve from 0.46413
885/885 ━━━━━━━━━━━━━━━━━━━━ 35s 40ms/step - accuracy: 0.4299 - loss: 1.4501 - val_accuracy: 0.4610 - val_loss: 1.3771 - learning_rate: 1.0000e-04
Epoch 39/40
885/885 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.4309 - loss: 1.4497
Epoch 39: val_accuracy did not improve from 0.46413
885/885 ━━━━━━━━━━━━━━━━━━━━ 41s 40ms/step - accuracy: 0.4309 - loss: 1.4497 - val_accuracy: 0.4566 - val_loss: 1.3817 - learning_rate: 1.0000e-04
Epoch 40/40
884/885 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 0.4292 - loss: 1.4460
Epoch 40: val_accuracy did not improve from 0.46413
885/885 ━━━━━━━━━━━━━━━━━━━━ 37s 42ms/step - accuracy: 0.4292 - loss: 1.4460 - val_accuracy: 0.4629 - val_loss: 1.3617 - learning_rate: 1.0